In [0]:
# Setup notebook parameters for catalog, batch number, and row limits
dbutils.widgets.text("catalog", "meridian_t12") 
dbutils.widgets.text("batch_number", "1")
dbutils.widgets.text("rows_per_batch", "2000")

catalog = dbutils.widgets.get("catalog")
batch_number = dbutils.widgets.get("batch_number")
rows = int(dbutils.widgets.get("rows_per_batch"))

volume_path = f"/Volumes/{catalog}/landing/raw_files"

from pyspark.sql.functions import col, lit, when, rand

# Function to inject bad data (null keys and duplicates) for Silver to catch
def inject_bad_data(df, primary_key):
    dupes = df.limit(5)
    df_corrupted = df.withColumn(
        primary_key, 
        when(rand() > 0.98, lit(None)).otherwise(col(primary_key))
    )
    return df_corrupted.union(dupes)

# Process Fact Tables (Disjoint batches per run)
facts = {"bookings": "booking_id", "payments": "payment_id"}
for table, pk in facts.items():
    df = spark.read.table(f"samples.wanderbricks.{table}")
    
    # Simulate disjoint batches by sampling with a seed based on the batch number
    df_batch = df.sample(withReplacement=False, fraction=0.3, seed=int(batch_number)).limit(rows)
    df_final = inject_bad_data(df_batch, pk)
    
    out_path = f"{volume_path}/{table}/batch_{batch_number}/"
    df_final.write.format("json").mode("overwrite").save(out_path)

# Process Dimension Tables (Land once during batch 1)
if batch_number == "1":
    dims = {"properties": "property_id", "destinations": "destination_id", "users": "user_id", "hosts": "host_id"}
    for table, pk in dims.items():
        df = spark.read.table(f"samples.wanderbricks.{table}")
        df_final = inject_bad_data(df, pk)
        
        out_path = f"{volume_path}/{table}/batch_1/"
        df_final.write.format("json").mode("overwrite").save(out_path)

elif batch_number == "4":  # to verify scd type 2
    print("Simulating base price changes for SCD Type 2...")
    df_props = spark.read.table("samples.wanderbricks.properties")
    
    # Increase the base_price by 10% to trigger history tracking
    df_props_changed = df_props.withColumn("base_price", col("base_price") * 1.1)
    
    out_path = f"{volume_path}/properties/batch_{batch_number}/"
    df_props_changed.write.format("json").mode("overwrite").save(out_path)


print(f"Batch {batch_number} landing complete.")